# Recherche textuelle texte -> image

Notebook extrait de `Recherche_textuelle-Clustering.ipynb`. Il conserve la partie recherche hybride: captions BLIP, embeddings OpenCLIP, Qdrant, BM25, RRF et LLM judge. Les sorties ont ?t? retir?es pour faciliter le versioning.

## 0) Setup & performance (GPU, batch, cache)


In [ ]:
import os, glob, time, math, json, hashlib, re
import numpy as np
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

BATCH_IMG = 64 if DEVICE == "cuda" else 16
BATCH_TXT = 256 if DEVICE == "cuda" else 64
NUM_WORKERS = 0  # Windows: souvent 0 est plus stable
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)


## 1) Dataset / Images


In [ ]:
IMAGE_DIR = "data/images/val2017"
paths = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.*")))
print("Nombre d'images:", len(paths))
paths[:5]


## 2) Modèles : BLIP (caption) + OpenCLIP (embeddings)

En production :
- captions + embeddings sont calculés **en background** lors de l’upload (job queue).
- la recherche temps réel ne calcule que l’embedding du **prompt**.


In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

BLIP_NAME = "Salesforce/blip-image-captioning-base"
processor = BlipProcessor.from_pretrained(BLIP_NAME)
blip = BlipForConditionalGeneration.from_pretrained(BLIP_NAME).to(DEVICE).eval()


In [ ]:
import open_clip

CLIP_MODEL = "ViT-L-14"
CLIP_PRETRAIN = "openai"

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(CLIP_MODEL, pretrained=CLIP_PRETRAIN)
clip_model = clip_model.to(DEVICE).eval()
clip_tokenizer = open_clip.get_tokenizer(CLIP_MODEL)

with torch.no_grad():
    dummy = torch.randn(1,3,224,224).to(DEVICE)
    dim = clip_model.encode_image(dummy).shape[-1]
print("OpenCLIP dim:", dim)


## 3) Cache local (éviter de recalculer captions/embeddings)


In [ ]:
CACHE_DIR = "data/artifacts"
os.makedirs(CACHE_DIR, exist_ok=True)

CAPTIONS_PATH = os.path.join(CACHE_DIR, "captions_blip.json")
IMG_EMB_PATH  = os.path.join(CACHE_DIR, "emb_img.npy")
CAP_EMB_PATH  = os.path.join(CACHE_DIR, "emb_cap.npy")
META_PATH     = os.path.join(CACHE_DIR, "meta.json")

def file_signature(file_paths, max_files=200):
    sample = file_paths[:max_files]
    sig = []
    for p in sample:
        try:
            sig.append((os.path.basename(p), os.path.getsize(p)))
        except:
            sig.append((os.path.basename(p), None))
    return hashlib.md5(json.dumps(sig).encode()).hexdigest()

DATA_SIG = file_signature(paths)
DATA_SIG


## 4) Génération captions BLIP (batch, GPU si dispo)


In [ ]:
from PIL import Image
from tqdm import tqdm

def blip_caption_images(image_paths, batch_size=8):
    caps = []
    for i in tqdm(range(0, len(image_paths), batch_size)):
        batch = image_paths[i:i+batch_size]
        imgs = [Image.open(p).convert("RGB") for p in batch]
        inputs = processor(images=imgs, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = blip.generate(**inputs, max_new_tokens=30)
        batch_caps = processor.batch_decode(out, skip_special_tokens=True)
        caps.extend([c.strip() for c in batch_caps])
    return caps

captions = None
if os.path.exists(CAPTIONS_PATH) and os.path.exists(META_PATH):
    meta = json.load(open(META_PATH,"r",encoding="utf-8"))
    if meta.get("data_sig") == DATA_SIG:
        captions = json.load(open(CAPTIONS_PATH,"r",encoding="utf-8"))
        print(" captions loaded from cache")

if captions is None:
    captions = blip_caption_images(paths, batch_size=8 if DEVICE=="cuda" else 4)
    json.dump(captions, open(CAPTIONS_PATH,"w",encoding="utf-8"), ensure_ascii=False, indent=2)
    json.dump({"data_sig": DATA_SIG}, open(META_PATH,"w",encoding="utf-8"), indent=2)
    print("captions generated & cached")

captions[:5]


## 5) Embeddings OpenCLIP (batch + autocast GPU)


In [ ]:
def l2_normalize(x, eps=1e-12):
    import numpy as np
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + eps)

@torch.no_grad()
def embed_images(image_paths, batch_size=BATCH_IMG):
    import numpy as np
    vecs = []
    for i in tqdm(range(0, len(image_paths), batch_size)):
        batch = image_paths[i:i+batch_size]
        imgs = [clip_preprocess(Image.open(p).convert("RGB")) for p in batch]
        imgs = torch.stack(imgs).to(DEVICE)

        if DEVICE == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                v = clip_model.encode_image(imgs)
        else:
            v = clip_model.encode_image(imgs)

        vecs.append(v.float().cpu().numpy())
    return l2_normalize(np.vstack(vecs))

@torch.no_grad()
def embed_texts(texts, batch_size=BATCH_TXT):
    import numpy as np
    vecs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        tok = clip_tokenizer(batch).to(DEVICE)

        if DEVICE == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                v = clip_model.encode_text(tok)
        else:
            v = clip_model.encode_text(tok)

        vecs.append(v.float().cpu().numpy())
    return l2_normalize(np.vstack(vecs))

img_vecs = cap_vecs = None
if os.path.exists(IMG_EMB_PATH) and os.path.exists(CAP_EMB_PATH) and os.path.exists(META_PATH):
    meta = json.load(open(META_PATH,"r",encoding="utf-8"))
    if meta.get("data_sig") == DATA_SIG:
        img_vecs = np.load(IMG_EMB_PATH)
        cap_vecs = np.load(CAP_EMB_PATH)
        print(" embeddings loaded from cache")

if img_vecs is None or cap_vecs is None:
    img_vecs = embed_images(paths)
    cap_vecs = embed_texts(captions)
    np.save(IMG_EMB_PATH, img_vecs)
    np.save(CAP_EMB_PATH, cap_vecs)
    print(" embeddings generated & cached")

img_vecs.shape, cap_vecs.shape


## 6) Qdrant (2 vecteurs nommés) + upsert batch


In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

QDRANT_URL = os.environ.get("QDRANT_URL", "http://localhost:6333")
COLLECTION = "photos_coco_poc"

client = QdrantClient(url=QDRANT_URL)

if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)

client.create_collection(
    collection_name=COLLECTION,
    vectors_config={
        "img": VectorParams(size=img_vecs.shape[1], distance=Distance.COSINE),
        "cap": VectorParams(size=cap_vecs.shape[1], distance=Distance.COSINE),
    }
)

def upsert_batch(ids, batch_img, batch_cap, payloads):
    pts = []
    for pid, v_img, v_cap, payload in zip(ids, batch_img, batch_cap, payloads):
        pts.append(PointStruct(
            id=int(pid),
            vector={"img": v_img.tolist(), "cap": v_cap.tolist()},
            payload=payload
        ))
    client.upsert(collection_name=COLLECTION, points=pts)

BATCH_UPSERT = 512
for i in tqdm(range(0, len(paths), BATCH_UPSERT)):
    batch_paths = paths[i:i+BATCH_UPSERT]
    ids = list(range(i, i+len(batch_paths)))
    payloads = [{"path": p, "filename": os.path.basename(p), "caption": captions[i+j]} for j,p in enumerate(batch_paths)]
    upsert_batch(ids, img_vecs[i:i+len(batch_paths)], cap_vecs[i:i+len(batch_paths)], payloads)

print(" Upsert done:", len(paths))


## 7) Recherche hybride (Vector + BM25) → RRF (K large pour le recall)

Production :
- k_vec / k_bm25 : 200–1000
- fused_top : 300–2000 (selon album)
- sortir **all_relevant** via seuil sémantique
- LLM uniquement pour **highlights** (Top 30–50)


In [ ]:
from rank_bm25 import BM25Okapi
from qdrant_client import models as qmodels

def tokenize_simple(s: str):
    return [w.lower() for w in re.findall(r"[a-zA-Z]{2,}", s)]

bm25 = BM25Okapi([tokenize_simple(c) for c in captions])

def bm25_search(query: str, top_k: int = 300):
    scores = bm25.get_scores(tokenize_simple(query))
    idx = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in idx]

@torch.no_grad()
def embed_query(query: str):
    return embed_texts([query], batch_size=1)[0]

def qdrant_vector_search(query_vec: np.ndarray, vector_name: str, top_k: int = 300):
    vec = query_vec.tolist()

    # New API: query_points(query=<vector>, using=<vector_name>)
    if hasattr(client, "query_points"):
        res = client.query_points(
            collection_name=COLLECTION,
            query=vec,                
            using=vector_name,        
            limit=top_k,
            with_payload=True
        )
        return [(int(p.id), float(p.score)) for p in res.points]

    # Old API fallback
    res = client.search(
        collection_name=COLLECTION,
        query_vector=(vector_name, vec),
        limit=top_k,
        with_payload=True
    )
    return [(int(p.id), float(p.score)) for p in res]


def rrf_fusion(rank_lists, k=60):
    scores = {}
    for lst in rank_lists:
        for rank, (doc_id, _) in enumerate(lst, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

def get_payload_map(ids):
    points = client.retrieve(COLLECTION, ids=ids, with_payload=True)
    return {int(p.id): p.payload for p in points}

def hybrid_search(query: str, k_vec=400, k_bm25=400, fused_top=700, rrf_k=60):
    qv = embed_query(query)
    v_img = qdrant_vector_search(qv, "img", top_k=k_vec)
    v_cap = qdrant_vector_search(qv, "cap", top_k=k_vec)
    t_bm  = bm25_search(query, top_k=k_bm25)

    fused = rrf_fusion([v_img, v_cap, t_bm], k=rrf_k)[:fused_top]
    ids = [doc_id for doc_id,_ in fused]
    payload_map = get_payload_map(ids)

    candidates = []
    for doc_id, rrf_score in fused:
        p = payload_map.get(doc_id, {})
        candidates.append({
            "id": doc_id,
            "rrf_score": float(rrf_score),
            "caption": p.get("caption",""),
            "filename": p.get("filename",""),
            "path": p.get("path","")
        })
    return candidates, qv

query = "people doing outdoor activities"
candidates, qv = hybrid_search(query)
len(candidates), candidates[0]


In [ ]:
def cosine(a, b):
    return float(np.dot(a, b))

def select_all_relevant(query_vec, candidate_ids, img_vecs, cap_vecs, min_sim=0.23):
    kept = []
    for cid in candidate_ids:
        sim = max(cosine(query_vec, img_vecs[cid]), cosine(query_vec, cap_vecs[cid]))
        if sim >= min_sim:
            kept.append((cid, sim))
    kept.sort(key=lambda x: x[1], reverse=True)
    return kept

all_relevant = select_all_relevant(qv, [c["id"] for c in candidates], img_vecs, cap_vecs, min_sim=0.23)
print("All relevant:", len(all_relevant))
all_relevant[:10]


In [ ]:
id_to_path = {c["id"]: c["path"] for c in candidates}


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import math

def show_images(
    ranked_ids,
    id_to_path,
    cols=5,
    max_images=25,
    title="Relevant images (pre-LLM)"
):
    ranked_ids = ranked_ids[:max_images]
    n = len(ranked_ids)
    rows = math.ceil(n / cols)

    plt.figure(figsize=(cols * 3, rows * 3))
    plt.suptitle(title, fontsize=16)

    for i, (cid, score) in enumerate(ranked_ids):
        try:
            img = Image.open(id_to_path[cid]).convert("RGB")
        except Exception as e:
            print(f"⚠️ Error loading {cid}: {e}")
            continue

        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.title(f"#{i+1} | sim={score:.2f}", fontsize=9)
        plt.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
show_images(
    all_relevant,
    id_to_path,
    cols=5,
    max_images=30,
    title="People are engaging in outdoor activities – semantic filter"
)


## 7.1) LLM Judge robuste (garder le contexte : concepts requis) + cards homogènes


In [ ]:
import json
import ollama
from typing import List, Dict

SYSTEM_PROMPT = """You are a strict ranking + filtering engine for a professional photo-search platform.

You do NOT see images. You only see captions and metadata.

Your job:
1) Extract the user's intent and convert it into REQUIRED CONCEPT GROUPS.
2) Re-rank candidates based ONLY on provided captions/metadata.
3) Filter out candidates that clearly violate the required concept groups.
4) Output STRICT JSON matching the schema.

Key rules:
- Do NOT invent visual details.
- Use ONLY caption/metadata evidence.
- If evidence is weak/ambiguous: lower confidence (do NOT hallucinate).
- Use RRF only as a tie-breaker.
- reason <= 20 words each; summary <= 40 words.
- Output JSON only (no markdown, no extra text).

IMPORTANT: REQUIRED concepts can be GROUPS with AND/OR.
Example:
Query: "people dancing at a wedding"
Required concept groups:
- Group A (must): ["people"]
- Group B (must): ["dancing" OR "celebrating"]
Context boosters (optional): ["wedding", "bride", "groom", "reception"]
"""

def build_user_prompt(query: str, candidates: List[Dict]) -> str:
    lines = []
    for rank, c in enumerate(candidates, start=1):
        cap = (c.get("caption") or "").replace("\n", " ").strip()[:260]
        lines.append(
            f'{rank}. id={c["id"]} | rrf_score={c.get("rrf_score", 0):.4f} | caption="{cap}"'
        )
    candidates_block = "\n".join(lines)

    return f"""User query:
"{query}"

Task (follow strictly):

Step A) Extract:
- required_concept_groups: list of groups.
  Each group must be satisfied.
  Inside a group, you may use OR alternatives.
- optional_boosters: concepts that improve ranking if present.
- negative_constraints (if implied): concepts that should NOT appear.

Step B) For each candidate:
- Determine group_match: does the caption support EACH required group?
- If any required group is missing, label MUST be "Not relevant" and confidence <= 0.25.
- If all required groups match but evidence is weak, use "Somewhat relevant" with lower confidence.

Step C) Ranking criteria (in order):
1) Required group coverage (hard constraint)
2) Overall thematic match to query
3) Specificity (more specific captions > generic)
4) Optional boosters present
5) RRF score as tie-breaker only

Candidates:
{candidates_block}

Return STRICT JSON with this schema:
{{
  "query": string,
  "required_concept_groups": [
    {{
      "group_name": string,
      "must_have_any": [string]
    }}
  ],
  "optional_boosters": [string],
  "negative_constraints": [string],
  "final_ranking": [
    {{
      "id": integer,
      "label": "Highly relevant" | "Relevant" | "Somewhat relevant" | "Not relevant",
      "reason": string,
      "confidence": number
    }}
  ],
  "filtered_out_ids": [integer],
  "summary": string
}}

Constraints:
- final_ranking MUST include ALL candidates exactly once.
- filtered_out_ids MUST contain every id labeled "Not relevant".
- JSON must be valid (double quotes, no trailing commas).
"""


In [ ]:
import json
import ollama
from typing import List, Dict

def llm_judge(query: str, candidates: List[Dict], model: str = "llama3.1:8b", max_candidates: int = 60):
    """
    Calls Ollama LLM to extract required concept groups + rerank + filter.
    Runs only on top `max_candidates` to control latency.
    """
    cands = candidates[:max_candidates]
    user_prompt = build_user_prompt(query, cands)

    resp = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        options={
            "temperature": 0.1,
            "top_p": 0.9
        }
    )

    text = resp["message"]["content"].strip()

    # Robust JSON extraction
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        s = text.find("{")
        e = text.rfind("}")
        if s != -1 and e != -1 and e > s:
            return json.loads(text[s:e+1])
        raise


In [ ]:
query = "people doing outdoor activities"

#  si tu veux, envoie seulement les candidats filtrés (all_relevant / top fused)
judge_out = llm_judge(query, candidates, model="llama3.1:8b", max_candidates=60)

judge_out["required_concept_groups"], judge_out["optional_boosters"][:5]


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

def resize_cover(img, size=(320,320)):
    return ImageOps.fit(img, size, method=Image.Resampling.LANCZOS, centering=(0.5,0.5))

def show_ranked_cards(judge_out, id_to_path, cols=4, tile_size=(320,320), top_k=12):
    keep_labels = {"Highly relevant","Relevant","Somewhat relevant"}
    ranking = [r for r in judge_out["final_ranking"] if r["label"] in keep_labels][:top_k]
    if not ranking:
        print("No relevant images.")
        return
    rows = (len(ranking)+cols-1)//cols
    plt.figure(figsize=(cols*4.2, rows*4.6))
    for i,item in enumerate(ranking):
        img = Image.open(id_to_path[item["id"]]).convert("RGB")
        img = resize_cover(img, tile_size)
        plt.subplot(rows, cols, i+1)
        plt.imshow(img); plt.axis("off")
        plt.title(f"#{i+1} • {item['label']} • {item['confidence']:.2f}", fontsize=11)
    plt.tight_layout(); plt.show()

id_to_path = {c["id"]: c["path"] for c in candidates}
show_ranked_cards(judge_out, id_to_path, top_k=50)
